In [31]:
import pandas as pd
import numpy as np
from core.cleaning import detect_stale_prices, detect_outliers, detect_missing_dates
from core.returns import log_returns

Build a realistic dirty dataset first: a DataFrame of daily close prices for two tickers over 10 days that has at least one of each of the following planted in it:

* A missing date (gap in the date index)
* A NaN value
* A stale price (same value repeated across consecutive days)
* An obvious bad print (price that's clearly wrong — use your judgment on what that looks like)
* Set a datetime index

Clean starting data set with random seeding

In [2]:
dates = pd.date_range('2026-05-08', periods=11, freq='B')

df = []

for ticker, (low, high) in {'CL': (75, 85), 'NG': (2.0, 3.0)}.items():
    df.append(pd.DataFrame({'date': dates, 'ticker': ticker, 
                            'close': np.random.uniform(low, high, 
                                                       size=len(dates))}))

df = pd.concat(df).set_index('date').sort_index()

print(df)

           ticker      close
date                        
2026-05-08     CL  76.303333
2026-05-08     NG   2.353946
2026-05-11     CL  79.790184
2026-05-11     NG   2.876558
2026-05-12     CL  75.763016
2026-05-12     NG   2.594924
2026-05-13     CL  84.572216
2026-05-13     NG   2.373344
2026-05-14     CL  82.373090
2026-05-14     NG   2.243052
2026-05-15     CL  79.898406
2026-05-15     NG   2.948518
2026-05-18     CL  79.842232
2026-05-18     NG   2.592576
2026-05-19     CL  80.216524
2026-05-19     NG   2.652715
2026-05-20     NG   2.423163
2026-05-20     CL  82.723785
2026-05-21     NG   2.836492
2026-05-21     CL  76.532309
2026-05-22     CL  81.549309
2026-05-22     NG   2.278252


* A missing date (gap in the date index)


In [3]:
df = df.drop(pd.Timestamp('2026-05-20'))
print(df)

           ticker      close
date                        
2026-05-08     CL  76.303333
2026-05-08     NG   2.353946
2026-05-11     CL  79.790184
2026-05-11     NG   2.876558
2026-05-12     CL  75.763016
2026-05-12     NG   2.594924
2026-05-13     CL  84.572216
2026-05-13     NG   2.373344
2026-05-14     CL  82.373090
2026-05-14     NG   2.243052
2026-05-15     CL  79.898406
2026-05-15     NG   2.948518
2026-05-18     CL  79.842232
2026-05-18     NG   2.592576
2026-05-19     CL  80.216524
2026-05-19     NG   2.652715
2026-05-21     NG   2.836492
2026-05-21     CL  76.532309
2026-05-22     CL  81.549309
2026-05-22     NG   2.278252


* A NaN value


In [4]:
df.loc[[pd.Timestamp('2026-05-18')], ['close']] = None
print(df)

           ticker      close
date                        
2026-05-08     CL  76.303333
2026-05-08     NG   2.353946
2026-05-11     CL  79.790184
2026-05-11     NG   2.876558
2026-05-12     CL  75.763016
2026-05-12     NG   2.594924
2026-05-13     CL  84.572216
2026-05-13     NG   2.373344
2026-05-14     CL  82.373090
2026-05-14     NG   2.243052
2026-05-15     CL  79.898406
2026-05-15     NG   2.948518
2026-05-18     CL        NaN
2026-05-18     NG        NaN
2026-05-19     CL  80.216524
2026-05-19     NG   2.652715
2026-05-21     NG   2.836492
2026-05-21     CL  76.532309
2026-05-22     CL  81.549309
2026-05-22     NG   2.278252


* A stale price (same value repeated across consecutive days)


In [5]:
# first set the date to NaN
df.loc[[pd.Timestamp('2026-05-14')], ['close']] = None
# then create a sub-dataframe of just consecutive days and ffill the close 
# column
df.loc[
    [pd.Timestamp('2026-05-13'), pd.Timestamp('2026-05-14')], 
    ['close']] = df.loc[
                        [pd.Timestamp('2026-05-13'), pd.Timestamp('2026-05-14')], 
                        ['ticker', 'close']].groupby(by='ticker').ffill()
print(df)


           ticker      close
date                        
2026-05-08     CL  76.303333
2026-05-08     NG   2.353946
2026-05-11     CL  79.790184
2026-05-11     NG   2.876558
2026-05-12     CL  75.763016
2026-05-12     NG   2.594924
2026-05-13     CL  84.572216
2026-05-13     NG   2.373344
2026-05-14     CL  84.572216
2026-05-14     NG   2.373344
2026-05-15     CL  79.898406
2026-05-15     NG   2.948518
2026-05-18     CL        NaN
2026-05-18     NG        NaN
2026-05-19     CL  80.216524
2026-05-19     NG   2.652715
2026-05-21     NG   2.836492
2026-05-21     CL  76.532309
2026-05-22     CL  81.549309
2026-05-22     NG   2.278252


* An obvious bad print (price that's clearly wrong — use your judgment on what that looks like)

In [6]:
# update Nat Gas
ng_mask = (df.index == pd.Timestamp('2026-05-21')) & (df['ticker'] == 'NG')
df.loc[ng_mask,['close']] = df.loc[ng_mask,['close']]*100
# update Crude
cl_mask = (df.index == pd.Timestamp('2026-05-21')) & (df['ticker'] == 'CL')
df.loc[cl_mask,['close']] = df.loc[cl_mask,['close']]*100
print(df)

           ticker        close
date                          
2026-05-08     CL    76.303333
2026-05-08     NG     2.353946
2026-05-11     CL    79.790184
2026-05-11     NG     2.876558
2026-05-12     CL    75.763016
2026-05-12     NG     2.594924
2026-05-13     CL    84.572216
2026-05-13     NG     2.373344
2026-05-14     CL    84.572216
2026-05-14     NG     2.373344
2026-05-15     CL    79.898406
2026-05-15     NG     2.948518
2026-05-18     CL          NaN
2026-05-18     NG          NaN
2026-05-19     CL    80.216524
2026-05-19     NG     2.652715
2026-05-21     NG   283.649151
2026-05-21     CL  7653.230927
2026-05-22     CL    81.549309
2026-05-22     NG     2.278252



* Set a datetime index

In [7]:
print(df.index.dtype)

datetime64[us]


In [8]:
detect_stale_prices(df)

date
2026-05-08    False
2026-05-08    False
2026-05-11    False
2026-05-11    False
2026-05-12    False
2026-05-12    False
2026-05-13    False
2026-05-13    False
2026-05-14     True
2026-05-14     True
2026-05-15    False
2026-05-15    False
2026-05-18    False
2026-05-18    False
2026-05-19    False
2026-05-19    False
2026-05-21    False
2026-05-21    False
2026-05-22    False
2026-05-22    False
Name: close, dtype: bool

In [9]:
detect_outliers(df, threshold=2.0)

date
2026-05-08    False
2026-05-08    False
2026-05-11    False
2026-05-11    False
2026-05-12    False
2026-05-12    False
2026-05-13    False
2026-05-13    False
2026-05-14    False
2026-05-14    False
2026-05-15    False
2026-05-15    False
2026-05-18    False
2026-05-18    False
2026-05-19    False
2026-05-19    False
2026-05-21     True
2026-05-21     True
2026-05-22    False
2026-05-22    False
Name: close, dtype: bool

In [10]:
detect_missing_dates(df)

DatetimeIndex(['2026-05-20'], dtype='datetime64[us]', freq='B')

Build a DataFrame with at least two tickers and 10+ rows of price data, then write a single groupby + agg call that returns — for each ticker — the mean price, std dev, min, max, and number of observations.

In [19]:
dates = pd.date_range('2026-05-14', periods=10, freq='B')

df =[]

for ticker, (low, high) in {'CL': (80.0, 90.0), 'NG': (2.0,3.0)}.items():
    df.append(
        pd.DataFrame(
            {'date': dates, 'ticker': ticker, 
                'close': np.random.uniform(low, high, size=len(dates))
            }
        )
    )

df = pd.concat(df).set_index('date').sort_index()

print(df)

           ticker      close
date                        
2026-05-14     CL  82.777502
2026-05-14     NG   2.190480
2026-05-15     CL  80.280186
2026-05-15     NG   2.232995
2026-05-18     CL  87.820713
2026-05-18     NG   2.803180
2026-05-19     CL  88.236236
2026-05-19     NG   2.019196
2026-05-20     CL  82.822818
2026-05-20     NG   2.954144
2026-05-21     CL  80.127208
2026-05-21     NG   2.277137
2026-05-22     CL  89.224051
2026-05-22     NG   2.744932
2026-05-25     NG   2.730976
2026-05-25     CL  81.665084
2026-05-26     NG   2.189676
2026-05-26     CL  86.216220
2026-05-27     CL  88.328137
2026-05-27     NG   2.578059


In [20]:
stats = df.groupby('ticker')['close'].agg(['mean', 'std', 'min', 'max', 'count'])
print(stats)

             mean       std        min        max  count
ticker                                                  
CL      84.749815  3.574975  80.127208  89.224051     10
NG       2.472078  0.325583   2.019196   2.954144     10


In [33]:
cl_close = df.loc[df['ticker'] == 'CL', ['close']]
weekly_cl_mean = cl_close.resample('W').agg(['mean'])
l_returns = log_returns(weekly_cl_mean)
print(l_returns)

               close
                mean
date                
2026-05-24  0.049268
2026-05-31 -0.002842
